# System ekspercki — głosowanie modeli (PetFinder.my)

Łączymy 4 wytrenowane modele (Random Forest, SVM, KNN, MLP) w system ekspercki oparty o **głosowanie** (`VotingClassifier`). Sprawdzamy hard voting oraz głosowanie ważone wynikami modeli i porównujemy je z pojedynczymi klasyfikatorami.

**Ważne:** RF trenowano na danych nieskalowanych, a SVM/KNN/MLP na danych skalowanych. Dlatego modele wrażliwe na skalę (SVM, KNN, MLP) owijamy w `Pipeline(StandardScaler, model)`, a do `VotingClassifier` podajemy dane **nieskalowane** — każdy pipeline skaluje sobie dane sam.

### Import bibliotek i konfiguracja

In [ ]:
from pathlib import Path

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
from sklearn.base import clone
from sklearn.ensemble import VotingClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)
from sklearn.model_selection import train_test_split

RANDOM_STATE = 1
np.random.seed(RANDOM_STATE)

ADOPTION_SPEED_LABELS = {
    0: '0–7',
    1: '8–30',
    2: '31–90',
    3: '>100',
}
CLASS_ORDER = sorted(ADOPTION_SPEED_LABELS.keys())
CLASS_NAMES = [ADOPTION_SPEED_LABELS[c] for c in CLASS_ORDER]

sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (10, 6)

### Sekcja 1 — Wczytanie danych i modeli

Wczytujemy dane, dzielimy je tak samo jak w poprzednich notebookach (`random_state=1`, `stratify=y`), skalujemy cechy i wczytujemy wszystkie wytrenowane modele. Kolejność kolumn ustawiamy zgodnie z `feature_names`, aby modele dostały cechy w tej samej kolejności co przy treningu.

In [ ]:
PROCESSED_DIR = '../data/processed'
MODELS_DIR = Path('../models')
INPUT_PATH = f'{PROCESSED_DIR}/train_clean_anomaly.csv'

df = pd.read_csv(INPUT_PATH)
if 'is_anomaly' in df.columns:
    df = df.drop(columns=['is_anomaly'])

y = df['AdoptionSpeed']
X = df.drop(columns=['AdoptionSpeed'])

# Wczytanie modeli i listy cech
rf_model = joblib.load(MODELS_DIR / 'rf_best.pkl')
svm_model = joblib.load(MODELS_DIR / 'svm_best.pkl')
knn_model = joblib.load(MODELS_DIR / 'knn_best.pkl')
nn_model = joblib.load(MODELS_DIR / 'nn_best.pkl')
feature_names = joblib.load(MODELS_DIR / 'feature_names.pkl')

# Ta sama kolejnosc kolumn co przy treningu
X = X[feature_names]

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=RANDOM_STATE,
    stratify=y,
)

# Skalowanie (potrzebne SVM/KNN/MLP); RF dziala na danych nieskalowanych
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

# Przypomnienie: accuracy kazdego modelu na zbiorze testowym
# RF -> dane nieskalowane, pozostale -> dane skalowane
base_acc = {
    'Random Forest': accuracy_score(y_test, rf_model.predict(X_test)),
    'SVM': accuracy_score(y_test, svm_model.predict(X_test_scaled)),
    'KNN': accuracy_score(y_test, knn_model.predict(X_test_scaled)),
    'MLP': accuracy_score(y_test, nn_model.predict(X_test_scaled)),
}

print('\nAccuracy pojedynczych modeli (punkt startowy):')
display(pd.DataFrame({'accuracy': base_acc}).round(4))

### Sekcja 2 — Hard Voting

Budujemy `VotingClassifier(voting='hard')` z 4 modeli. Każdy model „głosuje” na klasę, a wygrywa klasa z największą liczbą głosów. Modele wrażliwe na skalę (SVM, KNN, MLP) owijamy w `Pipeline(StandardScaler, model)`, żeby same skalowały dane — dzięki temu cały ensemble możemy uczyć na danych **nieskalowanych** (RF dostaje je wprost, bo drzewa skalowania nie wymagają; pipeline'y skalują u siebie). Każdy estymator jest niezależny, więc to, że RF nie ma `StandardScaler`, niczego nie psuje.

Uwaga: `VotingClassifier` klonuje i **trenuje od nowa** każdy estymator, korzystając z dostrojonych hiperparametrów wczytanych modeli.

In [ ]:
# SVM, KNN i MLP wymagaja skalowania -> Pipeline; RF dziala na danych surowych
svm_pipe = Pipeline([('scaler', StandardScaler()), ('model', clone(svm_model))])
knn_pipe = Pipeline([('scaler', StandardScaler()), ('model', clone(knn_model))])
nn_pipe = Pipeline([('scaler', StandardScaler()), ('model', clone(nn_model))])

estimators = [
    ('rf', clone(rf_model)),
    ('svm', svm_pipe),
    ('knn', knn_pipe),
    ('nn', nn_pipe),
]

hard_voting = VotingClassifier(estimators=estimators, voting='hard')
hard_voting.fit(X_train, y_train)

y_pred_hard = hard_voting.predict(X_test)

acc_hard = accuracy_score(y_test, y_pred_hard)
f1_hard = f1_score(y_test, y_pred_hard, average='macro')

print('=== Hard Voting ===')
print(classification_report(y_test, y_pred_hard, target_names=CLASS_NAMES))
print(f'Accuracy: {acc_hard:.4f}')
print(f'F1 macro: {f1_hard:.4f}')

cm_hard = confusion_matrix(y_test, y_pred_hard, labels=CLASS_ORDER)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_hard, annot=True, fmt='d', cmap='Blues',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Macierz pomyłek — Hard Voting')
plt.xlabel('Predykcja')
plt.ylabel('Rzeczywista klasa')
plt.tight_layout()
plt.show()

### Sekcja 3 — Weighted Voting

Tym razem głos każdego modelu **ważymy** jego dokładnością na zbiorze testowym: `waga = accuracy_modelu / suma_accuracy`. Lepsze modele mają większy wpływ na końcową decyzję.

In [ ]:
# Wagi proporcjonalne do accuracy pojedynczych modeli (kolejnosc jak w estimators: rf, svm, knn, nn)
acc_list = [base_acc['Random Forest'], base_acc['SVM'], base_acc['KNN'], base_acc['MLP']]
suma_acc = sum(acc_list)
weights = [a / suma_acc for a in acc_list]

wagi_tabela = pd.DataFrame({
    'Model': ['Random Forest', 'SVM', 'KNN', 'MLP'],
    'Accuracy': acc_list,
    'Waga': weights,
})
print('Wagi modeli:')
display(wagi_tabela.round(4))

weighted_voting = VotingClassifier(estimators=estimators, voting='hard', weights=weights)
weighted_voting.fit(X_train, y_train)

y_pred_weighted = weighted_voting.predict(X_test)

acc_weighted = accuracy_score(y_test, y_pred_weighted)
f1_weighted = f1_score(y_test, y_pred_weighted, average='macro')

print('\n=== Weighted Voting ===')
print(classification_report(y_test, y_pred_weighted, target_names=CLASS_NAMES))
print(f'Accuracy: {acc_weighted:.4f}')
print(f'F1 macro: {f1_weighted:.4f}')

cm_weighted = confusion_matrix(y_test, y_pred_weighted, labels=CLASS_ORDER)

plt.figure(figsize=(8, 6))
sns.heatmap(cm_weighted, annot=True, fmt='d', cmap='Greens',
            xticklabels=CLASS_NAMES, yticklabels=CLASS_NAMES)
plt.title('Macierz pomyłek — Weighted Voting')
plt.xlabel('Predykcja')
plt.ylabel('Rzeczywista klasa')
plt.tight_layout()
plt.show()

**Czy wagi poprawiły wynik?**

Porównaj `acc_weighted` z `acc_hard`. Ponieważ accuracy modeli są **zbliżone** (~0,38–0,42), wagi są niemal równe, więc głosowanie ważone zwykle daje wynik **bardzo podobny** do hard voting. Wyraźna poprawa pojawiłaby się dopiero, gdyby jeden model był znacznie lepszy od reszty.

### Sekcja 4 — Porównanie

Zestawiamy obie metody głosowania z pojedynczymi modelami, żeby zobaczyć, czy ensemble dał przewagę.

In [ ]:
porownanie = pd.DataFrame({
    'Accuracy': [acc_hard, acc_weighted],
    'F1-macro': [f1_hard, f1_weighted],
}, index=['Hard Voting', 'Weighted Voting'])

print('Porównanie metod głosowania:')
display(porownanie.round(4))

# Barplot: pojedyncze modele + obie metody glosowania
all_acc = pd.Series({
    **base_acc,
    'Hard Voting': acc_hard,
    'Weighted Voting': acc_weighted,
}).sort_values()

kolory = ['steelblue'] * len(base_acc) + ['crimson', 'darkorange']
# kolejnosc kolorow wg posortowanych wartosci
kolor_map = {**{k: 'steelblue' for k in base_acc},
             'Hard Voting': 'crimson', 'Weighted Voting': 'darkorange'}

plt.figure(figsize=(10, 6))
sns.barplot(x=all_acc.values, y=all_acc.index,
            hue=all_acc.index,
            palette=[kolor_map[k] for k in all_acc.index], legend=False)
for i, v in enumerate(all_acc.values):
    plt.text(v + 0.002, i, f'{v:.3f}', va='center')
plt.title('Accuracy: pojedyncze modele vs głosowanie')
plt.xlabel('Accuracy (zbiór testowy)')
plt.ylabel('Metoda / model')
plt.tight_layout()
plt.show()

**Które podejście wygrało i dlaczego?**

Głosowanie łączy mocne strony modeli i zwykle daje wynik **co najmniej tak dobry jak najlepszy pojedynczy model** (zwykle Random Forest), redukując ryzyko błędu jednego klasyfikatora. Hard i weighted voting są tu bardzo zbliżone, bo dokładności modeli są podobne — wagi nie zmieniają mocno większości głosów. Jeśli ensemble nie przebija RF, to dlatego, że słabsze modele (SVM, KNN) „dociągają” decyzję w dół; wtedy w praktyce wybralibyśmy sam Random Forest lub ensemble ważony silniej w stronę RF.

### Sekcja 5 — Predykcja na nowych danych

Tworzymy 5 przykładowych zwierząt o różnych profilach i sprawdzamy, jaką klasę `AdoptionSpeed` przewidzi system ekspercki (oba warianty głosowania). To symuluje działanie systemu na nowym ogłoszeniu.

In [ ]:
# 5 przykladowych rekordow (kolumny zgodne z feature_names)
# Kodowanie jak w danych PetFinder: Type 1=pies/2=kot; Vaccinated/Dewormed/Sterilized 1=tak,2=nie,3=nie wiadomo; Health 1=zdrowy
nowe_zwierzeta = pd.DataFrame([
    # mlody zaszczepiony kot z imieniem, darmowy, duzo zdjec -> raczej szybka adopcja
    {'Type': 2, 'Age': 3, 'Breed1': 266, 'Breed2': 0, 'Gender': 2, 'Color1': 1, 'Color2': 2, 'Color3': 0,
     'MaturitySize': 1, 'FurLength': 1, 'Vaccinated': 1, 'Dewormed': 1, 'Sterilized': 1, 'Health': 1,
     'Quantity': 1, 'Fee': 0, 'VideoAmt': 0, 'PhotoAmt': 6, 'HasName': 1, 'DescLength': 220},
    # starszy pies, wysoka oplata, bez imienia, malo zdjec -> raczej wolna adopcja
    {'Type': 1, 'Age': 72, 'Breed1': 307, 'Breed2': 0, 'Gender': 1, 'Color1': 2, 'Color2': 0, 'Color3': 0,
     'MaturitySize': 3, 'FurLength': 2, 'Vaccinated': 2, 'Dewormed': 2, 'Sterilized': 2, 'Health': 2,
     'Quantity': 1, 'Fee': 300, 'VideoAmt': 0, 'PhotoAmt': 1, 'HasName': 0, 'DescLength': 40},
    # mloda suczka, sredni profil
    {'Type': 1, 'Age': 6, 'Breed1': 189, 'Breed2': 0, 'Gender': 2, 'Color1': 1, 'Color2': 5, 'Color3': 0,
     'MaturitySize': 2, 'FurLength': 2, 'Vaccinated': 1, 'Dewormed': 1, 'Sterilized': 2, 'Health': 1,
     'Quantity': 1, 'Fee': 50, 'VideoAmt': 0, 'PhotoAmt': 3, 'HasName': 1, 'DescLength': 120},
    # grupa kociat (Quantity>1), darmowe, duzo zdjec
    {'Type': 2, 'Age': 2, 'Breed1': 266, 'Breed2': 0, 'Gender': 3, 'Color1': 1, 'Color2': 0, 'Color3': 0,
     'MaturitySize': 1, 'FurLength': 1, 'Vaccinated': 2, 'Dewormed': 1, 'Sterilized': 2, 'Health': 1,
     'Quantity': 4, 'Fee': 0, 'VideoAmt': 1, 'PhotoAmt': 8, 'HasName': 0, 'DescLength': 300},
    # dorosly kot, sredni wiek, umiarkowany profil
    {'Type': 2, 'Age': 24, 'Breed1': 264, 'Breed2': 0, 'Gender': 1, 'Color1': 3, 'Color2': 6, 'Color3': 0,
     'MaturitySize': 2, 'FurLength': 1, 'Vaccinated': 1, 'Dewormed': 1, 'Sterilized': 1, 'Health': 1,
     'Quantity': 1, 'Fee': 100, 'VideoAmt': 0, 'PhotoAmt': 2, 'HasName': 1, 'DescLength': 80},
])[feature_names]

pred_hard = hard_voting.predict(nowe_zwierzeta)
pred_weighted = weighted_voting.predict(nowe_zwierzeta)

wyniki = pd.DataFrame({
    'Zwierzę': [f'zwierzę {i+1}' for i in range(len(nowe_zwierzeta))],
    'Hard Voting': [ADOPTION_SPEED_LABELS[c] for c in pred_hard],
    'Weighted Voting': [ADOPTION_SPEED_LABELS[c] for c in pred_weighted],
}).set_index('Zwierzę')

print('Predykcje systemu eksperckiego dla nowych zwierząt (przedział dni do adopcji):')
display(wyniki)

**Interpretacja wyników**

- **Zwierzę 1** (młody, zaszczepiony kot, darmowy, dużo zdjęć) — profil sprzyjający szybkiej adopcji.
- **Zwierzę 2** (stary pies, wysoka opłata, bez imienia, jedno zdjęcie) — profil utrudniający adopcję, spodziewany dłuższy czas.
- Pozostałe rekordy mają profile pośrednie.

Oba warianty głosowania zwykle dają **podobne** predykcje (bo wagi są zbliżone). Różnice pojawiają się tam, gdzie modele „głosują” niejednomyślnie — wtedy wagi mogą przeważyć decyzję. Pamiętaj, że dokładność modeli to ~0,40, więc pojedyncze predykcje traktujemy orientacyjnie, a nie jako pewnik.

## Podsumowanie

- **Hard Voting** — każdy model ma równy głos; prosty i odporny sposób łączenia modeli.
- **Weighted Voting** — głosy ważone dokładnością modeli; przy zbliżonych accuracy daje wynik niemal identyczny jak hard voting.
- **Która metoda lepsza?** Porównaj tabelę `porownanie` — zwykle obie są bardzo blisko siebie i lądują w okolicach najlepszego pojedynczego modelu (Random Forest). Głosowanie ważone ma sens, gdy modele różnią się jakością; tutaj różnice są małe, więc przewaga jest niewielka.
- System ekspercki potrafi przyjąć **nowe ogłoszenie** i zwrócić przewidywany przedział czasu do adopcji (sekcja 5), korzystając z 4 połączonych modeli.

**Wniosek na obronę:** ensemble nie zawsze przebija najlepszy pojedynczy model — gdy klasyfikatory mają podobną (umiarkowaną) jakość, głosowanie stabilizuje wynik, ale nie tworzy nowej informacji. Ograniczeniem pozostaje informacyjność cech, a nie sposób łączenia modeli.

In [ ]:
print('Gotowe!')